# Week 3, day 1 (afternoon) — Worksheet 06 SOLUTIONS: dropping   (L02)

Every cell below was executed in the lab image (pandas 3.0.5) and the quoted
output is what it actually printed — including the error in Q10.

Question 2 is the one to re-read. It is the single most common Pandas mistake
there is, and it produces no error at all.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 06 — Dropping. Run this once.
import pandas as pd

people = pd.DataFrame(
    {
        "Name": ["Sara", "Ahmed", "Lina", "Omar"],
        "Age": [22, 21, 23, 25],
        "Address": ["Riyadh", "Jeddah", "Dammam", "Riyadh"],
        "Notes": [None, None, None, None],       # the empty column from the slide
    },
    index=["S1", "S2", "S3", "S4"],
)

print(people)
print()
print("shape:", people.shape)

PART A — the operation

### Question 1

`drop(index=["S2"])` -> Ahmed gone. `drop(columns=["Notes"])` -> `Notes` gone.

Both printed exactly what you would expect. Note that `index=` and
`columns=` name the axis explicitly, which is worth preferring over the
older `axis=0`/`axis=1` spelling — nobody misreads `columns=`.

In [ ]:
print("=== drop a row ===")
print(people.drop(index=["S2"]))
print()
print("=== drop a column ===")
print(people.drop(columns=["Notes"]))

### Question 2

`people` is **unchanged**: shape still `(4, 4)`, `Notes` still present, `S2` still present.

Q1 removed nothing. It computed two new frames, printed them, and threw
them away.

This is the most common Pandas mistake there is. `drop` **returns** a
modified copy and leaves the original alone — and because the printed
output in Q1 looked exactly right, there is nothing to alert you. You see
the column disappear on screen and reasonably conclude it is gone.

The failure arrives later and somewhere else: three cells further down, a
sum includes rows you thought you had removed. No exception, no warning,
just a number that is wrong by however much the dropped rows were worth.

In [ ]:
print(people)
print()
print("shape:", people.shape)
print("Notes still present:", "Notes" in people.columns)
print("S2 still present:   ", "S2" in people.index)

### Question 3

`trimmed` -> `(4, 3)`. `people` -> `(4, 4)`. -> the original keeps its `Notes` column.

Assigning the result is the whole fix. `trimmed` has the change; `people`
does not.

This is deliberate design, not an oversight. Returning a new object means
`people.drop(...)` can be used inside a larger expression without
corrupting the thing you are reading from, and it means two people
deriving different views from one frame cannot break each other. The cost
is that you must catch the result — every time.

In [ ]:
trimmed = people.drop(columns=["Notes"])
print("trimmed:", trimmed.shape, list(trimmed.columns))
print("people: ", people.shape, list(people.columns))

# drop RETURNS a new frame. The original is untouched. If you want the
# change to stick, you have to catch what comes back.

### Question 4

`drop(index=["S1","S4"], columns=["Notes","Address"])` -> a `(2, 2)` frame of Ahmed and Lina. -> original still `(4, 4)`.

Both axes in one call, and still a copy. Nothing about combining the two
arguments changes the return-a-copy rule.

In [ ]:
result = people.drop(index=["S1", "S4"], columns=["Notes", "Address"])
print(result)
print("shape:", result.shape)
print()
print("original untouched:", people.shape)

PART B — deciding what deserves to go

### Question 5

`isna().all()` -> `True`. `count()` -> `0`. -> then the drop is justified.

The slide's advice is 'the Notes column is empty, drop it'. That is
advice about a specific dataset, not a rule — and on a 4-row frame you can
see it is empty. On 400,000 rows you cannot.

`isna().all()` is the proof, and it takes one line. `count() == 0` says
the same thing a second way. Run one of them before dropping anything on
the grounds that it 'looks' empty, because a column that is 99.9% empty
looks identical in `head()` and is a completely different decision.

In [ ]:
print("all missing?  ", people["Notes"].isna().all())
print("values present:", people["Notes"].count())
print()
trimmed = people.drop(columns=["Notes"])
print(trimmed)

### Question 6

`dropna(axis=1, how="all")` -> drops `Notes` only. **Bare `dropna()` -> `(0, 4)`: every row gone.**

The default `dropna()` removes any row containing any missing value.
Every row here has an empty `Notes`, so every row qualifies, and you are
left with an empty frame that still has all four column headings.

This is how people accidentally delete their entire dataset in one line
and get no error for it. `dropna()` reads like 'tidy up the missing
values'; it actually means 'discard every record that is not completely
filled in'. On real data with a few sparse optional columns, that is
usually most of your rows.

The arguments are worth learning properly: `how="all"` requires the whole
row (or column) to be missing, and `subset=[...]` restricts the check to
columns you actually require.

In [ ]:
print("=== dropna(axis=1, how='all') ===")
print(people.dropna(axis=1, how="all"))
print()
print("=== dropna() with defaults ===")
print(people.dropna())
print()
print("shape after bare dropna():", people.dropna().shape)

### Question 7

Dropping the `>= 24` labels and filtering `< 24` -> the same three rows, `.equals()` is `True`.

They agree, and the filter is the better way to write it. `drop` has to
be handed a list of labels, so expressing a condition through it means
computing the rows you want to lose and then subtracting them — two
inversions to read instead of none.

Reach for `drop` when you know *which specific labels* must go, and for a
boolean filter when you know *what condition* the survivors satisfy.

In [ ]:
by_drop = people.drop(index=people[people["Age"] >= 24].index)
by_filter = people[people["Age"] < 24]
print(by_drop)
print()
print("same as filtering:", by_drop.equals(by_filter))

# Filtering is the clearer way to say this. drop() is for when you know
# WHICH labels to remove, not which condition to keep.

### Question 8

Chained -> a `(3, 3)` frame. -> original still `(4, 4)`.

Chaining works precisely because each call returns a new frame: the second
`drop` operates on the result of the first, and neither touches `people`.

The same property that makes Q2 a trap makes this safe.

In [ ]:
chained = people.drop(columns=["Notes"]).drop(index=["S1"])
print(chained)
print("shape:", chained.shape)
print()
print("original still:", people.shape)

### Question 9

`drop(columns=["Nickname"], errors="ignore")` -> `(4, 4)`, columns unchanged.

No error, and no change. That is what you asked for.

It is genuinely useful when clearing optional columns from files that may
or may not contain them. It is dangerous everywhere else, because it also
swallows typos: `drop(columns=["Adress"], errors="ignore")` quietly does
nothing and your real `Address` column survives into the output you were
trying to sanitise.

If you are dropping a column because it contains something that must not
be published, `errors="ignore"` is exactly the wrong flag.

In [ ]:
print(people.drop(columns=["Nickname"], errors="ignore").shape)
print("columns unchanged:", list(people.drop(columns=["Nickname"], errors="ignore").columns))

# Useful when clearing optional columns from files that may or may not have
# them. Dangerous everywhere else: drop(columns=["Adress"], errors="ignore")
# silently does nothing and your real Address column survives.

### Question 10

`drop(columns=["Nickname"])` -> **raises** `KeyError: "['Nickname'] not found in axis"`.

The default is strict, and the message names the label it could not find.

Put the sheet together and the shape of it is: Pandas is loud about
*structure* — a column that is not there stops you immediately — and silent
about *effect*. Dropping a column that exists, and then discarding the
result, is not an error at any level. The library will tell you when you
name something wrong and never tell you when you achieve nothing.

In [ ]:
print(people.drop(columns=["Nickname"]))